In [1]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np
import pandas as pd 
import joblib

In [2]:
X_user_train = pd.read_csv("../data/X_user_train_scaled.csv")
X_movie_train = pd.read_csv("../data/X_movie_train_scaled.csv")
y_train = pd.read_csv("../data/y_train_scaled.csv")

X_user_test = pd.read_csv("../data/X_user_test_scaled.csv")
X_movie_test = pd.read_csv("../data/X_movie_test_scaled.csv")
y_test = pd.read_csv("../data/y_test_scaled.csv")

scaler_users = joblib.load('../models/scaler_users.pkl')
scaler_movies = joblib.load('../models/scaler_movies.pkl')
scaler_y = joblib.load('../models/scaler_y.pkl')

In [3]:
# Train Linear Regression
X_train = pd.concat([X_user_train, X_movie_train], axis=1)
X_test = pd.concat([X_user_test, X_movie_test], axis=1)

lr_model = LinearRegression()
lr_model.fit(X_train, y_train)

# Predictions
y_train_pred = lr_model.predict(X_train)
y_train_pred_unscaled = scaler_y.inverse_transform(y_train_pred.reshape(-1, 1)).flatten()

y_test_pred = lr_model.predict(X_test)
y_test_pred_unscaled = scaler_y.inverse_transform(y_test_pred.reshape(-1, 1)).flatten()

#target y values
y_train_unscaled = scaler_y.inverse_transform(y_train).flatten()
y_test_unscaled = scaler_y.inverse_transform(y_test).flatten()

# Evaluation function
def evaluate_model(y_true, y_pred):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)

    return rmse, mae, r2

# Evaluate 
train_rmse, train_mae, train_r2 = evaluate_model(y_train_unscaled, y_train_pred_unscaled)
test_rmse, test_mae, test_r2 = evaluate_model(y_test_unscaled, y_test_pred_unscaled)

print("Linear Regression Results")

print(f"Training RMSE: {train_rmse:.4f}")
print(f"Training MAE:  {train_mae:.4f}")
print(f"Training R²:   {train_r2:.4f}")

print()

print(f"Test RMSE:       {test_rmse:.4f}")
print(f"Test MAE:        {test_mae:.4f}")
print(f"Test R²:         {test_r2:.4f}")

Linear Regression Results
Training RMSE: 0.8200
Training MAE:  0.6234
Training R²:   0.3695

Test RMSE:       0.8734
Test MAE:        0.6655
Test R²:         0.2874


BASELINE:

Linear Regression Results
Training RMSE: 0.8531
Training MAE:  0.6490
Training R²:   0.3590

Test RMSE:       0.9229
Test MAE:        0.7018
Test R²:         0.2531

In [7]:
train_data = pd.read_csv("../data/training_data.csv")
test_data = pd.read_csv("../data/test_data.csv")


In [8]:
from sklearn.metrics import ndcg_score


# Predictions are already unscaled above.
# Build evaluation DataFrames, keeping each row aligned with its user.
train_eval = pd.DataFrame({
    "userId": train_data["userId"].to_numpy(),
    "relevance": y_train_unscaled,
    "predicted_rating": y_train_pred_unscaled
})

test_eval = pd.DataFrame({
    "userId": test_data["userId"].to_numpy(),
    "relevance": y_test_unscaled,
    "predicted_rating": y_test_pred_unscaled
})

def mean_ndcg_at_10(eval_df):
    user_ndcgs = []

    for _, user_data in eval_df.groupby("userId"):
        # Skip users with no relevant variation in their ratings.
        if user_data["relevance"].nunique() <= 1:
            continue

        relevance = user_data["relevance"].to_numpy().reshape(1, -1)
        scores = user_data["predicted_rating"].to_numpy().reshape(1, -1)

        user_ndcgs.append(
            ndcg_score(relevance, scores, k=10)
        )

    return np.mean(user_ndcgs) if user_ndcgs else np.nan

print(f"Train NDCG@10: {mean_ndcg_at_10(train_eval):.4f}")
print(f"Test NDCG@10:  {mean_ndcg_at_10(test_eval):.4f}")

Train NDCG@10: 0.8656
Test NDCG@10:  0.8540
